In [3]:
import pandas as pd
import sqlite3
path = r'C:\Users\Ceren\Desktop\Olist_Project\\'
# 1. Verileri Oku (Dosyaların olduğu yolu belirt)
customers = pd.read_csv(path +'olist_customers_dataset.csv')
orders = pd.read_csv(path +'olist_orders_dataset.csv')
items = pd.read_csv(path +'olist_order_items_dataset.csv')
products = pd.read_csv(path +'olist_products_dataset.csv')

# 2. SQL Veritabanına Bağlan (Otomatik oluşturur)
conn = sqlite3.connect('OlistDatabase.db')

# 3. Verileri SQL Tablolarına Dönüştür
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
items.to_sql('items', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)

print("Veriler başarıyla SQL veritabanına (OlistDatabase.db) aktarıldı!")

Veriler başarıyla SQL veritabanına (OlistDatabase.db) aktarıldı!


In [4]:
import pandas as pd
import sqlite3
from sklearn.linear_model import LinearRegression
import numpy as np

# 1. Veritabanına bağlan ve zaman serisi verisini çek
conn = sqlite3.connect('OlistDatabase.db')
query = """
SELECT order_purchase_timestamp AS date
FROM orders
"""
df = pd.read_sql_query(query, conn)

# 2. Tarih formatını düzenle ve aylık satışları say
df['date'] = pd.to_datetime(df['date'])
df['month_year'] = df['date'].dt.to_period('M')
monthly_sales = df.groupby('month_year').size().reset_index(name='sales_count')

# 3. Tahmin Modeli Hazırlığı (Ayları sayıya çeviriyoruz: 1, 2, 3...)
monthly_sales['month_index'] = np.arange(len(monthly_sales))
X = monthly_sales[['month_index']] # Bağımsız değişken (Zaman)
y = monthly_sales['sales_count']    # Bağımlı değişken (Satış miktarı)

# 4. Modeli Eğit
model = LinearRegression()
model.fit(X, y)

# 5. Gelecek Ayı Tahmin Et
next_month_index = np.array([[len(monthly_sales)]])
prediction = model.predict(next_month_index)

print(f"Gelecek ay için tahmin edilen toplam sipariş miktarı: {int(prediction[0])}")

Gelecek ay için tahmin edilen toplam sipariş miktarı: 6477


C:\Users\Ceren\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [5]:
import sqlite3
import pandas as pd

# Veritabanına bağlan
conn = sqlite3.connect('OlistDatabase.db')

# Gerekli tabloları CSV olarak dışa aktar
pd.read_sql_query("SELECT * FROM orders", conn).to_csv('olist_orders_ready.csv', index=False)
pd.read_sql_query("SELECT * FROM items", conn).to_csv('olist_items_ready.csv', index=False)
pd.read_sql_query("SELECT * FROM products", conn).to_csv('olist_products_ready.csv', index=False)

# Tahmin sonucunu da küçük bir CSV yapalım
prediction_df = pd.DataFrame({'Metric': ['Next Month Forecast'], 'Value': [6477]})
prediction_df.to_csv('model_forecast.csv', index=False)

print("İşlem tamam! 'ready' uzantılı dosyaların klasöründe oluştu.")

İşlem tamam! 'ready' uzantılı dosyaların klasöründe oluştu.
